In [3]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

# --- locate the Millennium workbook wherever it is ---
here = Path.cwd()
hits = list(here.rglob("a-millennium-of-macroeconomic-data-for-the-uk.xlsx"))
if not hits:
    for parent in here.parents:
        hits = list(parent.rglob("a-millennium-of-macroeconomic-data-for-the-uk.xlsx"))
        if hits: break
xl = hits[0]
print("Using:", xl)

# --- primary balance £mn (A28 col 23) ---
a28 = pd.read_excel(xl, sheet_name="A28. Public Sector Borrowing", header=None)
pb = a28.iloc[8:, [0,23]].copy(); pb.columns=["year","psurp"]
pb["year"]=pd.to_numeric(pb["year"],errors="coerce"); pb["psurp"]=pd.to_numeric(pb["psurp"],errors="coerce")
pb=pb.dropna()

# --- nominal GDP £mn (A9 col 3) ---
a9 = pd.read_excel(xl, sheet_name="A9. Nominal GDP (A)", header=None)
g = a9.iloc[5:,[0,3]].copy(); g.columns=["year","gdp"]
g["year"]=pd.to_numeric(g["year"],errors="coerce"); g["gdp"]=pd.to_numeric(g["gdp"],errors="coerce"); g=g.dropna()

# --- debt as % GDP: BoE's own clean spliced series (A29 col 41) ---
a29 = pd.read_excel(xl, sheet_name="A29. The National Debt", header=None)
yr = a29.iloc[6:,0].apply(lambda s:int(str(s)[:4]) if str(s)[:4].isdigit() else np.nan)
debt = pd.to_numeric(a29.iloc[6:,41], errors="coerce")
db = pd.DataFrame({"year":yr,"debt":debt}).dropna(); db["year"]=db["year"].astype(int)

# --- merge; primary balance to % GDP; keep 1900-2015 ---
m = pb.merge(g, on="year").merge(db, on="year")
m["pb"] = m["psurp"]/m["gdp"]*100
m = m[(m.year>=1900)&(m.year<=2015)].sort_values("year").reset_index(drop=True)
m["debt_lag"] = m["debt"].shift(1)
m["war"] = m["year"].isin(list(range(1914,1919))+list(range(1939,1946))).astype(int)

# --- reaction function by era, CORRECT data ---
reg = m.dropna(subset=["debt_lag","pb"])
print("\nCORRECTED reaction function (clean BoE debt series):")
print("-"*55)
for lo,hi in [(1900,1918),(1919,1939),(1946,1973),(1974,1999),(2000,2015)]:
    s=reg.query("@lo<=year<=@hi")
    if len(s)>5:
        r=smf.ols("pb ~ debt_lag",data=s).fit()
        print(f"  {lo}-{hi} (n={len(s):3d}): beta={r.params['debt_lag']:+.4f}  p={r.pvalues['debt_lag']:.3f}")
noswar = smf.ols("pb ~ debt_lag", data=reg[reg.war==0]).fit()
print(f"\n  Ex-war full:      beta={noswar.params['debt_lag']:+.4f}  p={noswar.pvalues['debt_lag']:.3f}")

# save next to the workbook's Research folder OR just cwd
m.to_csv("long.csv", index=False)
print("\nsaved long.csv in:", Path.cwd())
print("\n2000-2015 debt values (should be ~29 to ~88):")
print(m[m.year>=2000][["year","debt","pb"]].round(1).to_string(index=False))

Using: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Research/Data/a-millennium-of-macroeconomic-data-for-the-uk.xlsx

CORRECTED reaction function (clean BoE debt series):
-------------------------------------------------------
  1900-1918 (n= 18): beta=-0.2708  p=0.000
  1919-1939 (n= 21): beta=+0.1182  p=0.003
  1946-1973 (n= 28): beta=+0.0170  p=0.039
  1974-1999 (n= 26): beta=-0.0513  p=0.328
  2000-2015 (n= 16): beta=-0.0664  p=0.005

  Ex-war full:      beta=+0.0336  p=0.000

saved long.csv in: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Research/Charts/Charts

2000-2015 debt values (should be ~29 to ~88):
 year  debt   pb
 2000  29.3  4.2
 2001  30.6  2.7
 2002  31.4  0.2
 2003  34.4 -0.8
 2004  35.7 -0.8
 2005  36.2 -0.9
 2006  36.7 -0.6
 2007  46.9 -0.2
 2008  61.8 -1.1
 2009  72.3 -5.2
 2010  76.2 -4.3
 2011  80.6 -2.0
 2012  83.6 -3.1
 2013  85.4 -1.8
 2014  86.0 -2.5
 2015  87.7 -1.7


In [5]:
a28 = pd.read_excel(xl, sheet_name="A28. Public Sector Borrowing", header=None)
pb = a28.iloc[8:, [0, 23]]   # col 0 = year, col 23 = primary surplus/deficit £mn
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [6]:
import pandas as pd, numpy as np
from pathlib import Path

m = pd.read_csv(list(Path.cwd().rglob("long.csv"))[0])   # your computed pb

xl_obr = list(Path.cwd().rglob("PSF_aggregates_databank_Mar_EFO.xlsx"))[0]
o = pd.read_excel(xl_obr, sheet_name="Aggregates (per cent of GDP)", header=None)
obr = o.iloc[4:, [1,11]].copy(); obr.columns=["year","pb_obr"]   # col 11 = primary balance % GDP, published
obr["year"]=obr["year"].apply(lambda v:int(str(v)[:4]) if str(v)[:4].isdigit() else np.nan)
obr["pb_obr"]=pd.to_numeric(obr["pb_obr"],errors="coerce")
obr=obr.dropna()

cmp = m.merge(obr, on="year")[["year","pb","pb_obr"]].copy()
cmp["diff"] = (cmp["pb"] - cmp["pb_obr"]).round(2)
print(cmp[cmp.year.isin([1980,1990,2000,2007,2010,2015])].round(2).to_string(index=False))
print("\nmean abs difference over overlap:", cmp["diff"].abs().mean().round(2), "pts")

IndexError: list index out of range